# Spark Session Initialization

Initialize the Spark Session used for all DataFrame operations in this notebook.

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import os
import sys

builder = ( SparkSession.builder \
    .appName("BGG Data Validation") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.LocalLogStore")
            
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Paths Configuration

Define all input and output paths used in this notebook.

In [ ]:
from pathlib import Path
from pyspark.sql.functions import (
    col, year, month, dayofmonth,
    round as spark_round, to_date,
    current_timestamp, when, round,
    months_between
)

PROJECT_ROOT = Path.cwd().parents[0]
DATA_PATH = PROJECT_ROOT / "data"

REFERENCE_PATH = DATA_PATH / "reference"
BRONZE_PATH = DATA_PATH / "bronze"
SILVER_PATH = DATA_PATH / "silver"

# Load Bronze-Level Delta Tables

Read raw transactional and generated bronze-level Delta tables into Spark DataFrames:
- sales
- games
- customers
- employees

In [ ]:
sales_df = spark.read.format("delta").load(str(BRONZE_PATH / "sales")) \
    .withColumn("date", to_date(col("sale_timestamp")))

In [ ]:
games_df = spark.read.format("delta").load(str(BRONZE_PATH / "bgg_games")) \
    .select(
        col("game_id"),
        col("names").alias("game_name"),
        col("avg_rating"),
        col("category"),
        col("mechanic")
    )

In [ ]:
customers_df = spark.read.format("delta").load(str(BRONZE_PATH / "customers")) \
    .select(
        col("customer_id"),
        col("first_name").alias("customer_first_name"),
        col("last_name").alias("customer_last_name"),
        col("country_id").alias("customer_country_id"),
        col("registration_date")
    )

In [ ]:
employees_df = spark.read.format("delta").load(str(BRONZE_PATH / "employees")) \
    .select(
        col("employee_id"),
        col("first_name").alias("employee_first_name"),
        col("last_name").alias("employee_last_name"),
        col("country_id").alias("employee_country_id"),
        col("hire_date")
    )

# Load Reference Data Delta Tables

Read preprocessed reference Delta tables for enrichment step:
- vendors
- delivery
- google analytics
- geography
- calendar

In [ ]:
calendar_df = spark.read.format("delta").load(str(REFERENCE_PATH / "calendar"))

In [ ]:
geo_df = spark.read.format("delta").load(str(REFERENCE_PATH / "geography")) \
    .select(
        "country_id",
        "country_name",
        "region_name",
        "continent_name"
    )

In [ ]:
vendors_df = spark.read.format("delta").load(str(REFERENCE_PATH / "vendors")) \
    .select(
        "vendor_id",
        "vendor_name",
        "vendor_country"
    )

In [ ]:
delivery_df = spark.read.format("delta").load(str(REFERENCE_PATH / "delivery")) \
    .select(
        "delivery_id",
        "delivery_company_name"
    )

In [ ]:
ga_df = spark.read.format("delta").load(str(REFERENCE_PATH / "google_analytics")) \
    .select(
        "ga_id",
        "ga_device_type",
        "ga_source_id"
    )

# Enrichment Step

Join and transform bronze tables with reference data to produce silver-level enriched delta table.

In [ ]:
sales_enriched_df = (
    sales_df
    .join(customers_df, "customer_id", "left")
    .join(employees_df, "employee_id", "left")
    .join(games_df, "game_id", "left")
    .join(vendors_df, "vendor_id", "left")
    .join(delivery_df, "delivery_id", "left")
    .join(ga_df, "ga_id", "left")
    .join(calendar_df, "date","left")
)

In [ ]:
geo_customer_df = geo_df.select(
    col("country_id"),
    col("country_name").alias("customer_country_name"),
    col("region_name").alias("customer_region_name"),
    col("continent_name").alias("customer_continent_name")
)

geo_employee_df = geo_df.select(
    col("country_id"),
    col("country_name").alias("employee_country_name"),
    col("region_name").alias("employee_region_name"),
    col("continent_name").alias("employee_continent_name")
)

In [ ]:
sales_enriched_df = (
    sales_enriched_df \
        .join(geo_customer_df.alias("geo_cust"), col("customer_country_id") == col("geo_cust.country_id"), 'left') \
        .join(geo_employee_df.alias("geo_emp"), col("employee_country_id") == col("geo_emp.country_id"), 'left') \
        .drop("country_id") \
)

In [ ]:
sales_enriched_df = (
    sales_enriched_df \
        .withColumn("total_revenue", round(col("quantity") * col("unit_price"),2))
        .withColumn("total_cost", round(col("quantity") * col("unit_cost"),2))
        .withColumn("total_profit", round(col("total_revenue") - col("total_cost"),2))
        .withColumn("profit_margin_pct",when(
                                            col("quantity") * col("unit_price") != 0,
                                            round(((col("quantity") * col("unit_price")) -
                                                    (col("quantity") * col("unit_cost"))
                                                  ) / (col("quantity") * col("unit_price")) * 100,2)).otherwise(0))
        .withColumn("is_new_customer",  months_between(col("sale_timestamp"), col("registration_date")) <= 1)
        .withColumn("silver_processed_timestamp", current_timestamp())
        .drop("ingestion_timestamp", "source_system")
)

# Save Enriched Silver Table

Persist the enriched silver-level DataFrame to Delta format for further analytics.

In [ ]:
from utils.data_io import save_to_silver

save_to_silver(sales_enriched_df, "sales_enriched")